In [ ]:
install open ai

In [ ]:
! pip install openai


import openai, os and set openai api key



In [ ]:
import os #for local setup when using venv
from openai import OpenAI

client = OpenAI(api_key="YOUR_OPENAI_API_KEY")

setting the system and giving it a prompt

In [ ]:
messages = [
    {"role": "system", "content": "you are a kind helpful assistant"}
]

running a infinite loop to keep chatting with agent and setting the code for chatbot

In [ ]:
while True:
  message=input("user: ")
  if message.lower() in ['exit','quit']:
    print("end")
    break
  else:
    #user input is added
    messages.append(
        {'role':'user','content': message}
    )
    #chat is sent
    chat=client.chat.completions.create(
        model="gpt-3.5-turbo",messages=messages, max_tokens = 50, temperature = 0.7
    )
    #chat is recieved
    reply=chat.choices[0].message.content
    print(f"chatGPT: {reply}")
    #again apendinding this to keep the chatbot in the context what's going on
    messages.append({'role':'assistant','content':reply})

user: 123


UnicodeEncodeError: 'ascii' codec can't encode character '\u0441' in position 14: ordinal not in range(128)

In [ ]:
import os
from openai import OpenAI

client=OpenAI("key")

#function to generate ai response
def generate_text(prompt):
  reply=client.chat.completions.create(
      model='gpt-3.5-turbo', messages=[{
          'role':'system', 'content':'you are a helpful assistant'
          },
          {'role':'user', 'content':prompt
      }],
      max_tokens=50, temperature=0.6
  )
  return reply.choices[0].message.content

#function for text translation
def translate_text(prompt, target='french'):
  reply=client.chat.completions.create(
      model='gpt-3.5-turbo', messages=[{
          'role':'system', 'content':'you are a translator'
          },
          {'role':'user', 'content':f'translate the following text to {target}:{prompt}'
          }],
          max_tokens=50, temperature=0.6
  )
  return reply.choices[0].message.content

#function to generate image
def image_generate(prompt):
  reply=client.images.generate(
      model='dall-e-2',
      prompt=prompt,
      n=1,
      size='1024x1024',
      response_format='b64_json'
  )
  import base64
  response=reply.data[0].b64_json
  with open('picture.png', 'wb') as f:
    f.write(base64.b64decode(response))
  print("image saved as picture.png")
  return 'picture.png'

#function to edit image and mask it
def edit_image_mask(prompt, image_path, mask_path):
  reply=client.images.edit(
      model='dall-e-2',
      prompt=prompt,
      image=open(image_path, 'rb'),
      mask=open(mask_path, 'rb'),
      n=1,
      size='1024x1024',
      response_format='b64_json'
  )

  import base64
  response=reply.data[0].b64_json
  with open('masked_image.png', 'wb') as f:
    f.write(base64.b64decode(response))
  print("masked image saved as masked_image.png")
  return 'masked_image.png'


#function to speech to text
def speech_to_text(audio_file):
  with open(audio_file, "rb") as f:
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=f
    )
  return transcript.text

#function text to speech
def text_to_speech(text):
  response = client.audio.speech.create(
    model="tts-1",
    voice="alloy",
    input=text
  )

  with open("speech.mp3", "wb") as f:
    f.write(response.content)

  print("Speech saved as speech.mp3")
  return 'speech.mp3'

#function using clip model to describe image
def describe_image(image_path):
  import base64
  with open(image_path, "rb") as img_file:
    # Read image content once for base64 encoding
    img_content = img_file.read()
    base64_image = base64.b64encode(img_content).decode('utf-8')

    response = client.chat.completions.create(
      model="gpt-4o-vision-preview",
      messages=[
          {
              "role": "user",
              "content": [
                  {"type": "text", "text": "Describe this image"},
                  {
                      "type": "image_url",
                      "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}
                  }
              ]
          }
      ],
      max_tokens=300,
    )
  return response.choices[0].message.content

In [ ]:
from sentence_transformer import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTrsansformer('all-MiniLM-L6-v2')
def senmantic_chunks(text, threshold=0.7):
  #breaking the text into sentences chunk and embedding it to derive semantic meanining
  sentences=text.split('.')
  embeddings=model.encode(sentences)

  chunks=[]
  current_chunk=sentences[0]

  for sentence in range(1, len(sentences)):
    sim=cosine_similarity([embeddings[0]], [embeddings[sentence]])[0][0]

    #chubking similar sentences in a group together
    if sim > threshold:
      current_chunk+=current_chunk.append(sentences[sentence])
    else:
      chunks.append(current_chunk)
      current_chunk=sentences[sentence]
  #chunking whatever is left in the current chunk
  chunks.append(" ".join(current_chunk))
  return chunks




ModuleNotFoundError: No module named 'sentence_transformer'